In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

 95%|█████████▌| 163M/170M [28:27<00:57, 136kB/s]

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 16 * 16, 100)
        self.fc2 = nn.Linear(100, 10)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))   # conv -> ReLU -> pool
        x = self.pool2(F.relu(self.conv2(x)))   # conv -> ReLU -> pool
        x = self.flatten(x)                     # flatten
        x = F.relu(self.fc1(x))                 # fc1, then ReLU
        x = self.fc2(x)                         # fc2 (no ReLU at the end)
        return x

In [ ]:
device = (
    torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)

model = CNN().to(device)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()                       # training mode
    total_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()           # clear the old gradients
        outputs = model(inputs)         # forward pass: 10 scores per image
        loss = loss_function(outputs, labels)   # how wrong were the scores?
        loss.backward()                 # work out how to adjust each weight
        optimizer.step()                # adjust the weights

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}  loss: {total_loss/len(train_loader):.4f}")

In [ ]:
model.eval()                            # evaluation mode
correct = 0
with torch.no_grad():                   # no learning, so skip gradient tracking
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        predictions = model(inputs).argmax(dim=1)   # index of the highest score
        correct += (predictions == labels).sum().item()

print(f"Test accuracy: {correct / len(test_dataset):.3f}")

In [ ]:
import os
os.makedirs("app", exist_ok=True)
torch.save(model.state_dict(), "app/cnn_cifar10.pt")

# In Colab, download the file, then put it in gentext-app/app/
try:
    from google.colab import files
    files.download("app/cnn_cifar10.pt")
except ImportError:
    pass
